# Module 3: Multi-Agent System with LangGraph

This notebook implements a sophisticated multi-agent system for Airbnb listings using:
- **LangGraph** for agent orchestration and state management
- **Specialized Agents** with distinct responsibilities
- **Supervisor Pattern** for intelligent task routing
- **Shared State** for coordination across agents

## Learning Objectives
- Understand multi-agent system architecture and benefits
- Design specialized agents with distinct responsibilities
- Implement agent orchestration using LangGraph
- Manage complex state across multiple agents
- Build a supervisor agent that routes tasks intelligently

## Step 1: Import Required Libraries and Setup Environment

In [ ]:
import os
import json
import operator
from typing import Annotated, TypedDict, Literal, List, Sequence, Dict, Any
from pymongo import MongoClient
from openai import OpenAI
from dotenv import load_dotenv

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Load environment variables
load_dotenv(override=True)

print("✅ Libraries imported and environment loaded")

## Step 2: Initialize Connections and LLM

In [ ]:
# Initialize OpenAI client
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Connect to DocumentDB
DOCUMENTDB_CONNECTION_STRING = os.getenv('DOCUMENTDB_CONNECTION_STRING')
mongo_client = MongoClient(DOCUMENTDB_CONNECTION_STRING)
db = mongo_client['db']
collection = db['listings']

# Initialize LangChain LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,
    api_key=os.getenv('OPENAI_API_KEY')
)

print("✅ Connections initialized")
print(f"📊 Database: {collection.count_documents({})} listings available")

In [ ]:
# Verify connection by fetching one document
test_doc = collection.find_one()
if test_doc:
    print(f"✅ Successfully retrieved document: {test_doc.get('name', 'Unknown')}")
    print(f"📊 Has embedding: {'descriptionVector' in test_doc}")
else:
    print("⚠️ No documents found. Please run Module 1 first to load data.")

## Step 3: Understanding Multi-Agent Architecture

### What are Multi-Agent Systems?

A multi-agent system uses multiple specialized AI agents that:
- Each handle specific tasks they're optimized for
- Communicate and coordinate with each other
- Work together to solve complex problems
- Make decisions about task delegation

### Why Use Multi-Agent Systems?

**Single Agent Approach:**
- ❌ One agent tries to do everything
- ❌ Complex prompts that confuse the model
- ❌ Difficult to maintain and debug

**Multi-Agent Approach:**
- ✅ Specialized agents with clear responsibilities
- ✅ Simpler, focused prompts per agent
- ✅ Easier to test and improve individual components
- ✅ More scalable and maintainable

## Step 4: Define the Shared State Schema

The state is shared across all agents and tracks the conversation. It includes:
- **messages**: Full conversation history
- **current_query**: The user's latest question
- **filters**: Extracted search criteria
- **search_results**: Listings found by the Search Agent
- **next_agent**: Which agent should run next
- **final_response**: The response to send back to the user

In [ ]:
class AgentState(TypedDict):
    """
    Shared state across all agents in the multi-agent system.
    """
    # Conversation history (automatically appended with add_messages)
    messages: Annotated[Sequence[HumanMessage | AIMessage], add_messages]
    
    # Current user query
    current_query: str
    
    # Extracted filters from the query
    filters: Dict[str, Any]
    
    # Search results from DocumentDB
    search_results: List[Dict[str, Any]]
    
    # Which agent should run next
    next_agent: str
    
    # Final response to user
    final_response: str
    
    # Session identifier
    session_id: str

print("✅ Agent state schema defined")

## Step 5: Create Helper Functions

We need embedding generation and vector search functions that our agents will use.

In [ ]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate a vector embedding for the given text using OpenAI.
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: A 1536-dimension vector representing the text
    """
    if not text or not isinstance(text, str):
        return None
    
    try:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

# Test the function
test_embedding = generate_embedding("cozy apartment near downtown")
print(f"✅ Embedding function ready")
print(f"📏 Test embedding dimensions: {len(test_embedding)}")

In [ ]:
def vector_search_with_filters(query: str, filters: Dict = None, limit: int = 5) -> List[Dict]:
    """
    Perform vector search with optional filters.
    
    Args:
        query: Search query text
        filters: Optional dictionary of filters
        limit: Maximum number of results
        
    Returns:
        List of matching listings
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query)
    
    if not query_embedding:
        return []
    
    # Build match conditions from filters
    match_conditions = {}
    
    if filters:
        if filters.get('bedrooms'):
            match_conditions['bedrooms'] = {"$gte": filters['bedrooms']}
        
        if filters.get('price_max'):
            match_conditions['price'] = {"$lte": filters['price_max']}
        
        if filters.get('price_min'):
            if 'price' in match_conditions:
                match_conditions['price']['$gte'] = filters['price_min']
            else:
                match_conditions['price'] = {"$gte": filters['price_min']}
        
        if filters.get('location'):
            match_conditions['address.market'] = {"$regex": filters['location'], "$options": "i"}
        
        if filters.get('property_type'):
            match_conditions['property_type'] = filters['property_type']
        
        if filters.get('amenities') and len(filters['amenities']) > 0:
            match_conditions['amenities'] = {"$all": filters['amenities']}
    
    # Build aggregation pipeline
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding,
                    "path": "descriptionVector",
                    "k": limit * 10  # Fetch more to account for filtering
                },
                "returnStoredSource": True
            }
        }
    ]
    
    # Add filter stage if we have conditions
    if match_conditions:
        pipeline.append({"$match": match_conditions})
    
    # Add projection and limit
    pipeline.extend([
        {
            "$project": {
                "_id": 1,
                "name": 1,
                "description": 1,
                "summary": 1,
                "property_type": 1,
                "bedrooms": 1,
                "beds": 1,
                "price": 1,
                "address": 1,
                "amenities": 1,
                "searchScore": {"$meta": "searchScore"}
            }
        },
        {"$limit": limit}
    ])
    
    results = list(collection.aggregate(pipeline))
    return results

# Test the function
test_results = vector_search_with_filters("cozy apartment", limit=3)
print(f"✅ Vector search function ready")
print(f"📊 Test search returned {len(test_results)} results")

## Step 6: Build Specialized Agents

We'll create four specialized agents:
1. **Filter Agent** - Extracts search filters from natural language
2. **Search Agent** - Performs vector search with filters
3. **Recommendation Agent** - Generates personalized recommendations
4. **Supervisor Agent** - Routes tasks to appropriate agents

### Agent 1: Filter Extraction Agent

This agent extracts structured filters from natural language queries.

In [ ]:
def filter_agent(state: AgentState) -> AgentState:
    """
    Extract search filters from the user's natural language query.
    
    Args:
        state: Current agent state
        
    Returns:
        Updated state with extracted filters
    """
    print("🔍 Filter Agent: Extracting filters from query...")
    
    query = state["current_query"]
    
    # Create prompt for filter extraction
    filter_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a filter extraction agent. Extract search filters from the user's query.

Return a JSON object with these fields (use null if not mentioned):
- bedrooms: number or null
- beds: number or null
- price_min: number or null
- price_max: number or null
- location: string (city/market name) or null
- property_type: string ("House", "Apartment", "Condominium", etc.) or null
- amenities: array of strings (e.g., ["Wifi", "Kitchen", "Parking", "Pet-friendly"])

Examples:
Query: "3 bedroom house in Chicago under $200"
Output: {{"bedrooms": 3, "property_type": "House", "location": "Chicago", "price_max": 200, "amenities": []}}

Query: "pet-friendly apartment with parking"
Output: {{"property_type": "Apartment", "amenities": ["Parking", "Pet-friendly"]}}

Query: "cozy place for a weekend getaway"
Output: {{"amenities": []}}

Only return valid JSON, no explanation."""),
        ("human", "{query}")
    ])
    
    chain = filter_prompt | llm | StrOutputParser()
    
    try:
        result = chain.invoke({"query": query})
        # Clean and parse the JSON
        result = result.strip()
        if result.startswith("```json"):
            result = result[7:]
        if result.startswith("```"):
            result = result[3:]
        if result.endswith("```"):
            result = result[:-3]
        
        filters = json.loads(result.strip())
        
        # Clean up null values
        filters = {k: v for k, v in filters.items() if v is not None and v != []}
        
        state["filters"] = filters
        state["next_agent"] = "search"
        
        print(f"   ✅ Extracted filters: {filters}")
        
    except json.JSONDecodeError as e:
        print(f"   ⚠️ Failed to parse filters, using empty: {e}")
        state["filters"] = {}
        state["next_agent"] = "search"
    
    return state

print("✅ Filter Agent created")

### Agent 2: Search Agent

This agent performs vector search using extracted filters.

In [ ]:
def search_agent(state: AgentState) -> AgentState:
    """
    Perform vector search with extracted filters.
    
    Args:
        state: Current agent state
        
    Returns:
        Updated state with search results
    """
    print("🔎 Search Agent: Performing vector search...")
    
    query = state["current_query"]
    filters = state.get("filters", {})
    
    # Perform vector search with filters
    results = vector_search_with_filters(query, filters, limit=5)
    
    state["search_results"] = results
    
    if results:
        state["next_agent"] = "recommendation"
        print(f"   ✅ Found {len(results)} listings")
    else:
        state["next_agent"] = "recommendation"
        print("   ⚠️ No listings found matching criteria")
    
    return state

print("✅ Search Agent created")

### Agent 3: Recommendation Agent

This agent analyzes search results and provides personalized recommendations.

In [ ]:
def recommendation_agent(state: AgentState) -> AgentState:
    """
    Generate personalized recommendations based on search results.
    
    Args:
        state: Current agent state
        
    Returns:
        Updated state with final response
    """
    print("💡 Recommendation Agent: Generating recommendations...")
    
    query = state["current_query"]
    filters = state.get("filters", {})
    results = state.get("search_results", [])
    
    if not results:
        state["final_response"] = """I couldn't find any listings matching your exact criteria. 

Here are some suggestions:
- Try expanding your search area
- Increase your budget slightly
- Reduce the number of required bedrooms
- Check different dates if you're flexible

Would you like me to search with adjusted criteria?"""
        state["next_agent"] = "end"
        return state
    
    # Format listings for the LLM
    listings_context = []
    for idx, result in enumerate(results, 1):
        listing_info = f"""
Listing {idx}:
- Name: {result.get('name', 'N/A')}
- Type: {result.get('property_type', 'N/A')}
- Location: {result.get('address', {}).get('market', 'N/A')}
- Bedrooms: {result.get('bedrooms', 'N/A')} | Beds: {result.get('beds', 'N/A')}
- Price: ${result.get('price', 'N/A')}/night
- Amenities: {', '.join(result.get('amenities', [])[:8])}
- Description: {result.get('summary', result.get('description', ''))[:200]}...
- Match Score: {result.get('searchScore', 0):.4f}
"""
        listings_context.append(listing_info)
    
    context = "\n".join(listings_context)
    
    # Create recommendation prompt
    recommendation_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a friendly and knowledgeable Airbnb recommendation agent.

Your task:
1. Analyze the retrieved listings against the user's query
2. Recommend the top 2-3 listings that best match their needs
3. Explain WHY each listing is a good match
4. Highlight key features (price, location, amenities)
5. Provide a clear, enthusiastic, conversational response

Guidelines:
- Be specific about what makes each listing suitable
- Mention the price and any standout amenities
- If all listings are similar quality, recommend based on best value
- End with a helpful question or next step suggestion
- Keep response concise (3-4 paragraphs total)
- Use emojis sparingly for visual appeal (⭐, 🏠, etc.)

User's Query: {query}
Extracted Filters: {filters}

Available Listings:
{context}"""),
        ("human", "Provide your recommendations based on the listings above.")
    ])
    
    chain = recommendation_prompt | llm | StrOutputParser()
    
    response = chain.invoke({
        "query": query,
        "filters": json.dumps(filters),
        "context": context
    })
    
    state["final_response"] = response
    state["next_agent"] = "end"
    
    print("   ✅ Generated personalized recommendations")
    
    return state

print("✅ Recommendation Agent created")

### Agent 4: Supervisor Agent

This agent determines which agent should handle the current task.

In [ ]:
def supervisor_agent(state: AgentState) -> AgentState:
    """
    Route tasks to the appropriate agent based on the current state.
    
    Args:
        state: Current agent state
        
    Returns:
        Updated state with next agent decision
    """
    print("📋 Supervisor: Analyzing task...")
    
    # If we have a query but no filters, start with filter extraction
    if state.get("current_query") and not state.get("filters"):
        state["next_agent"] = "filter"
        print("   → Routing to Filter Agent")
    
    # If we have filters but no search results, perform search
    elif state.get("filters") is not None and not state.get("search_results"):
        state["next_agent"] = "search"
        print("   → Routing to Search Agent")
    
    # If we have search results but no response, generate recommendations
    elif state.get("search_results") and not state.get("final_response"):
        state["next_agent"] = "recommendation"
        print("   → Routing to Recommendation Agent")
    
    # Otherwise, we're done
    else:
        state["next_agent"] = "end"
        print("   → Task complete")
    
    return state

print("✅ Supervisor Agent created")

## Step 7: Build the LangGraph Workflow

Now we'll create the state graph that connects all agents together.

In [ ]:
def route_next_agent(state: AgentState) -> str:
    """
    Determine the next node based on the state's next_agent field.
    
    Args:
        state: Current agent state
        
    Returns:
        Name of the next node to execute
    """
    next_agent = state.get("next_agent", "end")
    
    if next_agent == "filter":
        return "filter_agent"
    elif next_agent == "search":
        return "search_agent"
    elif next_agent == "recommendation":
        return "recommendation_agent"
    else:
        return END

print("✅ Routing function created")

In [ ]:
def create_agent_graph():
    """
    Create and compile the multi-agent graph.
    
    Returns:
        Compiled LangGraph application
    """
    # Create the graph
    workflow = StateGraph(AgentState)
    
    # Add nodes for each agent
    workflow.add_node("supervisor", supervisor_agent)
    workflow.add_node("filter_agent", filter_agent)
    workflow.add_node("search_agent", search_agent)
    workflow.add_node("recommendation_agent", recommendation_agent)
    
    # Set the entry point
    workflow.set_entry_point("supervisor")
    
    # Add conditional edges from supervisor
    workflow.add_conditional_edges(
        "supervisor",
        route_next_agent,
        {
            "filter_agent": "filter_agent",
            "search_agent": "search_agent",
            "recommendation_agent": "recommendation_agent",
            END: END
        }
    )
    
    # Add edges from each agent back to supervisor or to end
    workflow.add_conditional_edges(
        "filter_agent",
        route_next_agent,
        {
            "search_agent": "search_agent",
            END: END
        }
    )
    
    workflow.add_conditional_edges(
        "search_agent",
        route_next_agent,
        {
            "recommendation_agent": "recommendation_agent",
            END: END
        }
    )
    
    workflow.add_edge("recommendation_agent", END)
    
    # Compile the graph with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    
    return app

# Create the graph
agent_graph = create_agent_graph()

print("✅ Multi-agent graph created and compiled")

In [ ]:
# Optional: Visualize the workflow
try:
    from IPython.display import Image, display
    display(Image(agent_graph.get_graph().draw_mermaid_png()))
    print("📊 Agent workflow visualization displayed")
except Exception as e:
    print(f"ℹ️ Visualization requires additional dependencies: {e}")
    print("   Install with: pip install pygraphviz")

## Step 8: Run the Multi-Agent System

In [ ]:
def run_multi_agent_query(query: str, session_id: str = "default") -> dict:
    """
    Run a query through the multi-agent system.
    
    Args:
        query: User's natural language query
        session_id: Unique session identifier
        
    Returns:
        Final state with all agent outputs
    """
    print("=" * 80)
    print("🚀 Starting Multi-Agent Query")
    print("=" * 80)
    print(f"📝 Query: {query}")
    print(f"🔑 Session: {session_id}")
    print("=" * 80 + "\n")
    
    # Initialize state
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "current_query": query,
        "filters": None,
        "search_results": [],
        "next_agent": "supervisor",
        "final_response": "",
        "session_id": session_id
    }
    
    # Run the graph
    config = {"configurable": {"thread_id": session_id}}
    final_state = agent_graph.invoke(initial_state, config)
    
    print("\n" + "=" * 80)
    print("✅ Multi-Agent Processing Complete")
    print("=" * 80)
    
    return final_state

print("✅ Multi-agent query function created")

In [ ]:
# Test the system
result = run_multi_agent_query(
    query="Find me a pet-friendly 3 bedroom house in Chicago under $200",
    session_id="test_user_1"
)

# Display the response
print("\n🤖 Final Response:")
print("-" * 80)
print(result["final_response"])
print("-" * 80)

# Show extracted filters
print(f"\n🔍 Extracted Filters: {result['filters']}")
print(f"📊 Search Results Count: {len(result['search_results'])}")

## Step 9: Test Different Query Scenarios

In [ ]:
# Test with a different query
result2 = run_multi_agent_query(
    query="I need a cozy apartment with wifi for remote work",
    session_id="test_user_2"
)

print("\n🤖 Final Response:")
print("-" * 80)
print(result2["final_response"])
print("-" * 80)

In [ ]:
# Test with a luxury query
result3 = run_multi_agent_query(
    query="Luxury condo in Boston with parking and a view, budget $300 per night",
    session_id="test_user_3"
)

print("\n🤖 Final Response:")
print("-" * 80)
print(result3["final_response"])
print("-" * 80)

## Step 10: Add Conversation Continuity

Let's create a wrapper that handles multi-turn conversations.

In [ ]:
class MultiAgentChatManager:
    """
    Manages multi-turn conversations with the multi-agent system.
    """
    
    def __init__(self, agent_graph):
        self.agent_graph = agent_graph
        self.sessions: Dict[str, List[Dict]] = {}
    
    def chat(self, query: str, session_id: str, verbose: bool = True) -> str:
        """
        Process a chat message through the multi-agent system.
        
        Args:
            query: User's query
            session_id: Session identifier
            verbose: Whether to print agent actions
            
        Returns:
            AI response
        """
        if verbose:
            print("\n" + "=" * 60)
            print(f"🚀 Processing: {query[:50]}..." if len(query) > 50 else f"🚀 Processing: {query}")
            print("=" * 60)
        
        # Get or create session history
        if session_id not in self.sessions:
            self.sessions[session_id] = []
        
        # Build messages from history
        messages = []
        for exchange in self.sessions[session_id]:
            messages.append(HumanMessage(content=exchange["query"]))
            messages.append(AIMessage(content=exchange["response"]))
        messages.append(HumanMessage(content=query))
        
        # Initialize state
        initial_state = {
            "messages": messages,
            "current_query": query,
            "filters": None,
            "search_results": [],
            "next_agent": "supervisor",
            "final_response": "",
            "session_id": session_id
        }
        
        # Run the graph
        config = {"configurable": {"thread_id": session_id}}
        final_state = self.agent_graph.invoke(initial_state, config)
        
        response = final_state["final_response"]
        
        # Store in session history
        self.sessions[session_id].append({
            "query": query,
            "response": response,
            "filters": final_state.get("filters", {}),
            "results_count": len(final_state.get("search_results", []))
        })
        
        if verbose:
            print("\n✅ Processing complete")
        
        return response
    
    def get_session_history(self, session_id: str) -> List[Dict]:
        """Get the conversation history for a session."""
        return self.sessions.get(session_id, [])
    
    def clear_session(self, session_id: str):
        """Clear a session's history."""
        if session_id in self.sessions:
            del self.sessions[session_id]

# Initialize the chat manager
chat_manager = MultiAgentChatManager(agent_graph)

print("✅ Multi-agent chat manager initialized")

In [ ]:
# Test multi-turn conversation
session = "conversation_test"

print("\n" + "=" * 80)
print("💬 Multi-Agent Conversation Test")
print("=" * 80)

# First query
q1 = "I'm looking for a place in Denver for a ski trip"
r1 = chat_manager.chat(q1, session)
print(f"\n👤 User: {q1}")
print(f"\n🤖 Assistant:\n{r1}")

print("\n" + "-" * 80)

In [ ]:
# Follow-up query
q2 = "Do any of those have a hot tub?"
r2 = chat_manager.chat(q2, session)
print(f"\n👤 User: {q2}")
print(f"\n🤖 Assistant:\n{r2}")

print("\n" + "-" * 80)

In [ ]:
# Another follow-up
q3 = "What's the cheapest option with at least 2 bedrooms?"
r3 = chat_manager.chat(q3, session)
print(f"\n👤 User: {q3}")
print(f"\n🤖 Assistant:\n{r3}")

# Show session history
print("\n" + "=" * 80)
print(f"📊 Session History: {len(chat_manager.get_session_history(session))} exchanges")
print("=" * 80)

## Step 11: Test Various Scenarios

Let's test the multi-agent system with different types of queries.

In [ ]:
# Test different query scenarios
test_scenarios = [
    {
        "name": "Family Vacation",
        "query": "Family-friendly 4 bedroom house with a pool near beaches"
    },
    {
        "name": "Business Trip",
        "query": "Downtown apartment with fast wifi and workspace under $150"
    },
    {
        "name": "Romantic Getaway",
        "query": "Cozy cabin for couples with fireplace and mountain views"
    },
    {
        "name": "Budget Travel",
        "query": "Cheapest options in Chicago, just need a clean place to sleep"
    }
]

print("🧪 Testing Multi-Agent Scenarios\n")
print("=" * 80)

for i, scenario in enumerate(test_scenarios):
    session_id = f"scenario_{i}"
    print(f"\n📋 Scenario: {scenario['name']}")
    print(f"📝 Query: {scenario['query']}")
    print("-" * 40)
    
    response = chat_manager.chat(scenario['query'], session_id, verbose=False)
    
    # Show truncated response
    if len(response) > 400:
        print(f"🤖 Response: {response[:400]}...")
    else:
        print(f"🤖 Response: {response}")
    
    print("\n" + "=" * 80)

Change 

## 🔗 Explore Your Changes in the Application

Congratulations on building a multi-agent system! Now let's see how it powers the production application.

### Backend Implementation: `src/api/agents.py`

The agents module implements the same LangGraph architecture:

| Notebook Concept | Backend Implementation |
|-----------------|----------------------|
| `AgentState` TypedDict | `AgentState` in [agents.py](../src/api/agents.py#L42-L52) |
| `filter_agent()` | Filter extraction with JSON parsing |
| `search_agent()` | Uses `vector_search()` from search module |
| `recommendation_agent()` | Generates personalized responses |
| `supervisor_agent()` | Routes to appropriate specialist agents |
| LangGraph `StateGraph` | Same graph construction pattern |

**👉 Try it yourself:**
1. Open [src/api/agents.py](../src/api/agents.py) and find the `AgentState` class
2. Compare the agent tools with your implementations in Step 6
3. Notice how it gracefully degrades if LangGraph isn't available

### Architecture Comparison

```
Your Notebook                    Production Backend
─────────────────────────────────────────────────────
filter_agent()         →         create_search_tool()
search_agent()         →         apply_filters() tool
recommendation_agent() →         Agent graph with ToolNode
supervisor_agent()     →         Supervisor routing logic
MultiAgentChatManager  →         Integrated in chat endpoint
```

### Frontend Integration Points

The [ChatPanel](../src/frontend/src/components/ChatPanel.tsx) displays agent results:

- 🏠 **ListingCard**: Renders each listing from `response.listings`
- 🗺️ **MapView**: Shows listings on interactive map
- 📊 **ListingsPanel**: Side panel with full listing details

### Backend API Flow

```
User Query → /query_message endpoint
                ↓
         agents.py (if LangGraph available)
                ↓
         Filter Agent → Search Agent → Recommendation Agent
                ↓
         Structured response with listings + AI message
                ↓
         Frontend renders results
```

### 🚀 Run the Complete Multi-Agent System

Start the full stack application:

```bash
docker-compose up --build
```

Visit http://localhost:3000 and test complex queries that exercise all agents:

1. **Filter Agent**: "3 bedroom house in Chicago under $200"
   - Watch filters get extracted automatically

2. **Search Agent**: "pet-friendly apartment with parking"
   - Vector search + amenity filtering

3. **Recommendation Agent**: "romantic getaway for couples"
   - Semantic understanding + personalized suggestions

4. **Multi-turn**: Try follow-up questions like "What about the second one?"

### 🔧 Development Tips

- **Enable verbose logging** in the backend to see agent routing:
  ```python
  logging.basicConfig(level=logging.DEBUG)
  ```

- **Test individual agents** by importing them in a Python shell:
  ```python
  from src.api.agents import filter_agent, search_agent
  ```

- **Extend the system** by adding new specialized agents following the pattern you learned!